# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library. All dataset entities such as record sets, fields, and columns are referenced by their `@id` according to the Croissant specification for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata (use attribute access, not item/dict-access)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Cite As: {metadata.citeAs}")


## 2. Data Overview
Review the available record sets, fields, columns, and their `@id`s. This step helps us understand the dataset organization before extracting data.

We list the record sets and then enumerate their field and column `@id`s. All references are by canonical Croissant `@id` strings.

In [ ]:
from pprint import pprint

# List all available record sets in the dataset by their '@id' and print their fields
print("Available record sets (by @id):\n")
record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
for record_set_id in record_set_ids:
    print(f"- {record_set_id}")

if not record_set_ids:
    print("No record sets indexed at root. Attempting to find record set definitions in the schema.")
    # Sometimes record sets are at the top-level under 'recordSet', or as separate node entities
    # Below: Scan '@graph' for RecordSet definitions
croissant_json = dataset.metadata.to_json()
record_sets = []
if '@graph' in croissant_json:
    for node in croissant_json['@graph']:
        if node.get('@type') == 'cr:RecordSet':
            record_sets.append(node)

print("\nLocated record sets in the graph:")
overview = {}
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nRecord set: {rs_id}")
    overview[rs_id] = {}
    # Fetch fields
    field_ids = []
    # 'field' can be a list of dicts or list of @id strings
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, list):
            for field in fields:
                if isinstance(field, dict) and '@id' in field:
                    field_ids.append(field['@id'])
                elif isinstance(field, str):
                    field_ids.append(field)
        else:
            # Single field
            field_ids.append(fields if isinstance(fields, str) else fields.get('@id', fields))
    print(f"  Fields (@id): {field_ids}")
    overview[rs_id]['fields'] = field_ids
    # Fetch columns (usually attached to fields)
    all_column_ids = []
    if '@graph' in croissant_json:
        for node in croissant_json['@graph']:
            if node.get('@type') == 'cr:Field' and node.get('@id') in field_ids:
                columns = node.get('column', [])
                if isinstance(columns, list):
                    for col in columns:
                        if isinstance(col, dict) and '@id' in col:
                            all_column_ids.append(col['@id'])
                        elif isinstance(col, str):
                            all_column_ids.append(col)
                elif columns:
                    all_column_ids.append(columns if isinstance(columns, str) else columns.get('@id', columns))
    print(f"  Columns (@id): {all_column_ids}")
    overview[rs_id]['columns'] = all_column_ids

# We'll use overview for further extraction.
if not record_sets:
    print("No record sets found in '@graph'. Please check schema definition.")

# Show an example of the top-level record set @id we will use downstream
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    print(f"\nFirst main record set id: {example_record_set_id}")

## 3. Data Extraction
Load data from the record sets into pandas DataFrames for analysis. All record sets and their fields/columns are referenced by their canonical `@id` from the overview above.

In [ ]:
# Extract data from each record set found in the overview
# We use variable names matching the RecordSet @id for clarity

dataframes = dict()
main_record_set_ids = list(overview.keys())
for rs_id in main_record_set_ids:
    try:
        recs = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"Loaded record set '{rs_id}' (records: {len(df)})")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Display column names for each DataFrame
for rs_id, df in dataframes.items():
    print(f"\nColumns in record set '{rs_id}':")
    print(df.columns.tolist())
    print(df.head())
    # Just show columns and top 5 rows for the first one, then break for brevity
    break


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping. Replace the example field and record-set `@id`s below with the IDs discovered above.

In [ ]:
# EDA - Example using the first main record set and GAD-7 total score as a numeric field

# Use the main record set id and look for likely numeric fields (such as GAD-7 sum score id)
record_set_id = list(dataframes.keys())[0]
df = dataframes[record_set_id]

# For example, let's suppose the GAD-7 score column is '@gad7_sum' (replace with actual @id column as needed)
# For demonstration, identify all integer and float columns:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Numeric columns available: {numeric_cols}")
if numeric_cols:
    numeric_field_id = numeric_cols[0]  # Take first numeric field as example (GAD-7 or PHQ-9 sum likely)
else:
    numeric_field_id = None

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean()  # use mean as example threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (showing top 5):")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If there is a likely group/categorical field (e.g., '@gender' or similar found in the DataFrame columns), use it
    categorical_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field = ''
    if categorical_fields:
        group_field = categorical_fields[0]
    if group_field in df.columns and group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(name=f"{numeric_field_id}_mean")
        print(f"\nGrouped data by '{group_field}' (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found to demonstrate EDA.")

## 5. Visualization
Visualize distribution or relationships between fields using matplotlib or seaborn. Here we plot the distribution of the main numeric field and, if available, its breakdown by the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If there is a grouping field
    if group_field in df.columns and group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field)
        plt.xticks(rotation=30)
        plt.show()


## 6. Conclusion
In this notebook, we used `mlcroissant` to discover and load the FAIR² Mental Health Survey Dataset, referenced all entities by their canonical Croissant `@id`, and performed basic exploratory analysis and visualizations. You may adapt further EDA steps based on the specific fields of interest for your analysis or machine-learning tasks.